# AXI head-to-head, free version: the kept chat WITH COMPACTION against AXI, on the same job, on Gemini 3.5 Flash Lite
**What this is.** Agent tools shorten a chat when it gets long: the model is asked to summarise it, and the summary replaces it. This notebook runs that rival on the SAME job as test 22 (the dense service graph, seed 20261111, the same heavy turns) in two versions, `S` (generic summary at 32,000 tokens) and `V` (the summary writes out the complete current service graph), 300 steps each, every reply applied as it comes, and then runs AXI (`A`) on the same model and the same 300 steps. The predictions and the 14 reading rules were written before any call in `docs/LITE_COMPACTION_TEST.md` (hash-locked). The test can come out against AXI, and the page says so.

**This test costs nothing, and this notebook cannot turn billing on.** It uses the free key you already have, on **Gemini 3.5 Flash Lite** (your AI Studio page: 15 calls a minute, 500 calls a day). The notebook spaces its calls 6 seconds apart and stops by itself after 480 calls in a day. About 940 calls are needed in all: about two days. Everything finished is saved to Google Drive.

Run each cell **one at a time**, in order. When a cell stops with "STOPPED FOR NOW", wait for the time it says, open this notebook again, run Cells 2 to 5, then run the cell you were on again.

### Cell 1: put your key into Colab Secrets (once; no code to run)
Colab left bar -> key icon (*Secrets*) -> **Add new secret**. Name: `GOOGLE_API_KEY` (exactly). Value: your key from `aistudio.google.com` -> *API Keys*. Switch **Notebook access** on. Never paste the key into a cell or into the chat.

### Cell 2: get the code from GitHub (finish the update commands in Codespaces and `git push` first)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_lite_compaction.py"), "the code is not on GitHub yet: finish the update commands (git push) first"
assert os.path.exists("/content/axi/docs/LITE_COMPACTION_TEST.md"), "the page is not on GitHub yet: finish the update commands (git push) first"
print("code ok")

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: Connect / Allow). Your progress and every reply are kept there

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_lite_compaction")

### Cell 5: settings. Nothing to fill in: the key is the free one from Cell 1, and the notebook has no price and no budget

In [ ]:
import hashlib, json, time
from google.colab import userdata, files
from axi.experiments import gemini_long as G
from axi.experiments import gemini_compaction as C
from axi.experiments import gemini_lite_compaction as L
from axi.experiments.api_llm import QuotaStop, ServiceBusy

try: KEY = userdata.get("GOOGLE_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: a Secret named GOOGLE_API_KEY with Notebook access on.")
CACHE = "axi_gemini_lite_compaction_cache.jsonl"
print("reply file:", store.restore(CACHE))
llm, MODEL = L.build_lite_llm(KEY, store.path(CACHE), on_write=lambda: store.persist(CACHE))
META = dict(model=MODEL, thinking=None)
print("model:", MODEL, "| free key: calls spaced", L.MIN_INTERVAL, "s apart, at most", L.DAILY_CALLS, "calls a day")
RESET = "The free quota resets at midnight Pacific time. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do not run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        print(BUSY if isinstance(e, ServiceBusy) else RESET)
        return None
def save(res, name):
    p = store.path(name + ".json"); json.dump(res, open(p, "w")); store.persist(name + ".json"); return p
def persisting(name, inner):
    print("progress file:", store.restore(name + ".jsonl"))
    def cb(*a):
        inner(*a)
        if a[0] % 25 == 0 or a[0] == a[1]: store.persist(name + ".jsonl")
    return cb
def prog(t, n, r):
    if t % 25 == 0 or t == n:
        c = "summary" if "compaction" in r else "-"
        print(f"step {t}/{n} | prompt {r['prompt_tokens']:,} (cached {r['cache_tokens']:,}) | {r['seconds']:.1f} s | held exact {r['exact']} | reply exact {r['reply_exact']} | last step had a summary: {c}")

### Cell 6: ARM S, generic summary at 32,000 tokens. 300 steps. A line every 25 steps. If it stops (daily quota, busy server, tab closed), run Cells 2 to 5 and this cell again

In [ ]:
nameS = "gl_S_seed%d" % L.SEED
resS = go(lambda: L.run_arm(llm, "S", store.path(nameS + ".jsonl"), meta=META, progress=persisting(nameS, prog)))
if resS:
    print("stopped because:", resS["stopped"] or "the 300 steps were done")
    L.print_arm("ARM S", resS["rows"]); store.persist(nameS + ".jsonl"); files.download(save(resS, nameS))

### Cell 7: ARM V, the summary writes out the complete current graph, at 32,000 tokens. 300 steps. If it stops, run Cells 2 to 5 and this cell again

In [ ]:
nameV = "gl_V_seed%d" % L.SEED
resV = go(lambda: L.run_arm(llm, "V", store.path(nameV + ".jsonl"), meta=META, progress=persisting(nameV, prog)))
if resV:
    print("stopped because:", resV["stopped"] or "the 300 steps were done")
    L.print_arm("ARM V", resV["rows"]); store.persist(nameV + ".jsonl"); files.download(save(resV, nameV))

### Cell 8: ARM A, AXI: the host holds the exact state and shows it at every step. 300 steps. If it stops, run Cells 2 to 5 and this cell again

In [ ]:
nameA = "gl_A_seed%d" % L.SEED
resA = go(lambda: L.run_arm(llm, "A", store.path(nameA + ".jsonl"), meta=META, progress=persisting(nameA, prog)))
if resA:
    print("stopped because:", resA["stopped"] or "the 300 steps were done")
    L.print_arm("ARM A (AXI)", resA["rows"]); store.persist(nameA + ".jsonl"); files.download(save(resA, nameA))

### Cell 9: the READING. Replays every saved reply and every summary (a check that nothing was changed), prints the three arms and the 14 lines with yes / NO, and saves the report. Send me the output and the downloaded files

In [ ]:
arms = dict(S=resS, V=resV, A=resA); ok = {}
for k, res in arms.items():
    try: L.replay_arm(res["rows"], k); ok[k] = True
    except Exception as e: print("replay", k, ":", e); ok[k] = False
checks = L.verdict(arms, ok)
for k, res in arms.items(): L.print_arm("arm " + k, res["rows"])
L.print_verdict(checks)
for k in "SV":
    cs = C.compactions(arms[k]["rows"]); print(f"\narm {k}: {len(cs)} summaries; the first and the last (cut at 800 characters):")
    for c in (cs[:1] + cs[-1:]): print("  ->", c["text"][:800])
print("usage:", llm.usage())
files.download(save(dict(S=resS, V=resV, A=resA, checks=checks, replays=ok, model=MODEL), "gemini_lite_compaction_all"))

### Done
Files downloaded: the arm S, V and A files and `gemini_lite_compaction_all.json`. Upload `gemini_lite_compaction_all.json` to the chat. Nothing here says anything about any company's own tool or about another model: it measures Gemini 3.5 Flash Lite on this job.